# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E001-pipe-check-gold58** — three per-plane fluid-sensitive
models (frozen backbone + linear head) on the 58 gold-labeled studies (issue #6).
Requires the `WANDB_API_KEY` Kaggle secret.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
COMMIT = "main"  # TODO: pin to a SHA per run
%pip install -q "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee[train]"

from knee.series import SeriesType
from knee.train_gold import train_gold

In [ ]:
# Gold-58 prototype trains straight off the mounted competition data; the private
# mined-labels dataset joins here later (issue #2).
from pathlib import Path

COMP_ROOT = Path("/kaggle/input/rsna-knee-abnormality-detection")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(project="rsna-knee", config={"commit": COMMIT})
except Exception as exc:
    print(f"wandb disabled: {exc}")

In [ ]:
# E001: one specialist per fluid plane (strict typing — no fallback series; studies
# lacking a plane are skipped by that model). Non-fluid models are a contingency,
# not a plan — see docs/modeling understanding/optimization-levers.md.
SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
INPUT_SIZE = 224

# pipe_check_gold58: trains on the gold studies, must never be evaluated against them.
def checkpoint_path(series_type: SeriesType) -> Path:
    return Path(f"/kaggle/working/pipe_check_gold58_{series_type.value}.pt")

In [ ]:
# Competition DICOMs are pre-mounted read-only; ~58 series per model, minutes of GPU each.
results = []
for series_type in SERIES_TYPES:
    result = train_gold(COMP_ROOT, checkpoint_path(series_type), series_type=series_type, input_size=INPUT_SIZE)
    results.append(result)
    print(f"{series_type.value}: trained on {result.n_studies}, skipped {len(result.skipped)}")
    # In-sample only (trains on all gold rows): proves the features carry signal, nothing more.
    print(result.in_sample_auc)

In [ ]:
# Checkpoints are already in /kaggle/working, which persists as notebook output;
# publish all three as the knee-weights dataset so the inference notebook can attach them.
import math

if run is not None:
    for result in results:
        wandb.log(
            {
                f"in_sample_auc/{result.series_type.value}/{label}": auc
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    run.finish()